# From a hand-built data lake to a governed semantic layer

## Overview
This notebook walks a real-estate ESG portfolio dataset from raw client deliveries through to a
governed semantic layer an agent can answer questions against. It is the companion to the
Portfolio Copilot demo: the app shows the outcome, this shows the work.

## Why it is shaped this way
Many ESG data platforms follow a similar pattern: daily dumps from operational databases
into parquet files on object storage, every use case served by its own script, no warehouse,
no semantic layer, and only the engineering team able to query anything. The three raw tables
here stand in for that kind of lake, complete with the delivery problems platforms actually
face: inconsistent country spellings, floor areas in mixed units, and smart meters that report
intermittently.

## What you will see
- The transformation mechanics in **both SQL and Snowpark Python**, showing how the two
  compose naturally within a single pipeline
- How messy client deliveries get harmonised without losing the evidence of the mess
- How sparse and estimated consumption is flagged rather than quietly averaged away
- Why the CRREM pathway comparison is materialised at asset-year grain instead of computed on read
- How two intensity definitions coexist in one semantic model
- Client isolation proven by running the same query under three roles, in Python, with no
  filtering code in it
- An honest comparison against a hand-built lake and against a lakehouse, including where the
  argument stops

## Prerequisites
- `CUSTOM_DEMOS.DEEPKI` populated by `agent/01_create_tables.sql` and `agent/02_create_semantic_view.sql`
- A role with SELECT on that schema, and USAGE on `COMPUTE_WH`

## Estimated time: 15 minutes

In [ ]:
# === ALL IMPORTS ===
import logging
import pandas as pd
import plotly.express as px
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

session = get_active_session()
session.sql('USE SCHEMA CUSTOM_DEMOS.DEEPKI').collect()

logging.getLogger().setLevel(logging.INFO)
logger = logging.getLogger('deepki_demo')
logger.info(f'Connected to {session.get_current_database()}.{session.get_current_schema()}')

## 1. The raw layer, exactly as delivered

Three feeds, three different sets of problems. Nothing has been cleaned yet.

`RAW_ASSET_REGISTER` is what arrives from a client's own asset management system. Note the
country column: the same country appears as `Germany`, `GERMANY` and `DEUTSCHLAND` depending
on which feed the row came through. Floor area arrives in square metres for most clients and
square feet for others. Occupancy is missing wherever the sector does not track it.

In [ ]:
-- The delivery problem, made visible
SELECT country_raw,
       floor_area_unit,
       COUNT(*)                                        AS assets,
       COUNT(occupied_units)                           AS with_occupied_units,
       COUNT(*) - COUNT(occupancy_rate_pct)            AS missing_occupancy
FROM   RAW_ASSET_REGISTER
GROUP  BY country_raw, floor_area_unit
ORDER  BY assets DESC
LIMIT  15

Sector labels arrive in variant forms too, including French domain terms that mean the same
thing as the English ones. `EHPAD` is a French nursing home and `Bureaux` is an office.

In [ ]:
SELECT sector_raw, COUNT(*) AS assets
FROM   RAW_ASSET_REGISTER
GROUP  BY sector_raw
ORDER  BY assets DESC

## 2. Harmonising, without hiding the evidence

`DIM_ASSET` resolves all three problems. The important part is that the raw table is still
there: harmonisation is a transformation, not a destructive edit, so any figure can be traced
back to what the client actually sent.

In [ ]:
SELECT asset_country, asset_sector,
       COUNT(*)                        AS assets,
       ROUND(SUM(floor_area_m2))       AS total_m2,
       ROUND(AVG(floor_area_m2))       AS avg_m2
FROM   DIM_ASSET
GROUP  BY asset_country, asset_sector
ORDER  BY total_m2 DESC
LIMIT  12

A spot check that the unit conversion actually worked. Every asset delivered in square feet
should now hold a plausible square-metre value, roughly a tenth of the delivered number.

In [ ]:
SELECT r.source_asset_ref,
       r.floor_area_value  AS delivered_value,
       r.floor_area_unit   AS delivered_unit,
       a.floor_area_m2     AS harmonised_m2,
       ROUND(r.floor_area_value / a.floor_area_m2, 3) AS implied_ratio
FROM   RAW_ASSET_REGISTER r
JOIN   DIM_ASSET a ON a.asset_id = r.source_asset_ref
WHERE  r.floor_area_unit = 'sqft'
ORDER  BY r.source_asset_ref
LIMIT  5

### How `DIM_ASSET` is actually built

The queries above read the harmonised table. They do not show how it was produced, which is
the part that matters if you are the team who would have to maintain it. Here is the same
transformation in Snowpark, the Python DataFrame API.

This is deliberately shown in Python rather than SQL. Most data engineers and data
scientists working in ESG platforms write Python, and the idioms below — `select`, `when/otherwise`, `alias`,
`group_by`, `agg`, `join` — are the ones they already use in PySpark. The transferable skill
is the point: nobody has to become a SQL developer to own this pipeline.

In [ ]:
raw = session.table('RAW_ASSET_REGISTER')

# Country: three feeds spell the same country three ways.
country_fix = (
    F.when(F.upper(F.col('COUNTRY_RAW')) == F.lit('DEUTSCHLAND'), F.lit('Germany'))
     .when(F.upper(F.col('COUNTRY_RAW')) == F.lit('FR'),          F.lit('France'))
     .when(F.upper(F.col('COUNTRY_RAW')) == F.lit('UK'),          F.lit('United Kingdom'))
     .otherwise(F.initcap(F.col('COUNTRY_RAW')))
)

# Sector: French domain terms mean the same as the English ones.
sector_fix = (
    F.when(F.upper(F.col('SECTOR_RAW')) == F.lit('EHPAD'),        F.lit('Nursing home'))
     .when(F.upper(F.col('SECTOR_RAW')) == F.lit('BUREAUX'),      F.lit('Office'))
     .when(F.upper(F.col('SECTOR_RAW')) == F.lit('NURSING HOME'), F.lit('Nursing home'))
     .otherwise(F.initcap(F.col('SECTOR_RAW')))
)

# Units: one feed in six reports square feet.
area_m2 = (
    F.when(F.col('FLOOR_AREA_UNIT') == F.lit('sqft'),
           F.round(F.col('FLOOR_AREA_VALUE') / F.lit(10.7639), 2))
     .otherwise(F.col('FLOOR_AREA_VALUE'))
)

# EPC band to a consumption multiplier. A lookup table would be better in
# production; inline here so the whole rule is visible in one screen.
efficiency = (
    F.when(F.col('EPC_RATING') == F.lit('A'), F.lit(0.60))
     .when(F.col('EPC_RATING') == F.lit('B'), F.lit(0.70))
     .when(F.col('EPC_RATING') == F.lit('C'), F.lit(0.85))
     .when(F.col('EPC_RATING') == F.lit('D'), F.lit(1.00))
     .when(F.col('EPC_RATING') == F.lit('E'), F.lit(1.20))
     .when(F.col('EPC_RATING') == F.lit('F'), F.lit(1.45))
     .when(F.col('EPC_RATING') == F.lit('G'), F.lit(1.70))
     .otherwise(F.lit(1.10))
)

harmonised = raw.select(
    F.col('SOURCE_ASSET_REF').alias('ASSET_ID'),
    F.col('SOURCE_CLIENT_REF').alias('CLIENT_ID'),
    F.col('ASSET_LABEL').alias('ASSET_NAME'),
    country_fix.alias('ASSET_COUNTRY'),
    F.col('CITY').alias('ASSET_CITY'),
    sector_fix.alias('ASSET_SECTOR'),
    area_m2.alias('FLOOR_AREA_M2'),
    F.col('OCCUPIED_UNITS'),
    F.col('OCCUPANCY_RATE_PCT'),
    F.col('HEATING_SYSTEM'),
    F.col('EPC_RATING'),
    F.col('BUILD_YEAR'),
    efficiency.alias('EFFICIENCY_FACTOR'),
)

harmonised.filter(F.col('ASSET_COUNTRY') == 'Germany').select(
    'ASSET_ID', 'ASSET_COUNTRY', 'ASSET_SECTOR', 'FLOOR_AREA_M2'
).show(5)

Nothing has executed on the client. `harmonised` is a lazy plan, and Snowpark has compiled it
to SQL that runs in the warehouse. You can read the SQL it generated, which matters when you
are debugging someone else's pipeline at 2am:

In [ ]:
# The DataFrame API is a SQL generator. No data moved to this notebook to
# build it, and there is no separate execution engine to reason about.
print(harmonised.queries['queries'][0][:600])

And the Python path produces exactly the table the SQL script built. Same logic, same result,
expressed in whichever language the person maintaining it prefers:

In [ ]:
built = session.table('DIM_ASSET')
compare_cols = ['ASSET_ID', 'ASSET_COUNTRY', 'ASSET_SECTOR', 'FLOOR_AREA_M2']

differing = harmonised.select(compare_cols).subtract(built.select(compare_cols)).count()

print(f'Snowpark rows:      {harmonised.count():,}')
print(f'SQL-built rows:     {built.count():,}')
print(f'Rows differing:     {differing}')
assert differing == 0, 'The Python and SQL definitions have diverged'
print('\nThe two definitions agree.')

### Mixing SQL and Python in the same pipeline

One of the strengths of Snowpark is that SQL and Python are not separate worlds. You can
start a step in SQL, capture the result as a DataFrame, continue transforming in Python,
and write the result back, all in the same session and the same transaction. Nothing moves
to a client or crosses an engine boundary.

This matters for real pipelines because some steps are cleaner in SQL (window functions,
recursive CTEs, pivots) while others are cleaner in Python (conditional logic trees,
calling external libraries, iterative calculations). Being able to pick the right tool per
step, without a serialisation boundary between them, is the difference between an elegant
pipeline and a duct-taped one.

In [ ]:
# Start in SQL: a CTE that ranks assets by floor area within each sector.
# Window functions are one of those things SQL does more cleanly than Python.
ranked_sql = session.sql('''
    SELECT asset_id, asset_sector, asset_country, floor_area_m2,
           RANK() OVER (PARTITION BY asset_sector ORDER BY floor_area_m2 DESC) AS size_rank
    FROM   DIM_ASSET
''')

# Continue in Python: filter, transform, add a derived column.
# The DataFrame returned by session.sql() is a first-class Snowpark DataFrame,
# the same object the table() API returns.
top_assets = (
    ranked_sql
    .filter(F.col('SIZE_RANK') <= 5)
    .with_column('SIZE_CATEGORY',
        F.when(F.col('FLOOR_AREA_M2') > 50000, F.lit('Very large'))
         .when(F.col('FLOOR_AREA_M2') > 20000, F.lit('Large'))
         .otherwise(F.lit('Medium')))
    .select('ASSET_SECTOR', 'ASSET_COUNTRY', 'FLOOR_AREA_M2', 'SIZE_RANK', 'SIZE_CATEGORY')
    .sort('ASSET_SECTOR', 'SIZE_RANK')
)

top_assets.show()

The SQL ran a window function (clean in SQL, verbose in Python). The Python added conditional
logic (clean in Python, verbose in SQL CASE expressions). Neither step moved data to the
notebook: `session.sql()` returns a lazy Snowpark DataFrame, and the `.filter()` / `.with_column()`
chain extends the execution plan. The warehouse executes one combined query.

This composability also works in reverse. You can build a DataFrame in Python, then hand its
generated SQL to a downstream SQL step via `df.queries`, or write it to a table that a SQL
task reads. The two languages share the same engine, the same governance, and the same
transaction.

In [ ]:
# The reverse direction: build in Python, inspect the SQL it generates.
# This is how you debug a Snowpark pipeline at the SQL level.
generated_sql = top_assets.queries['queries'][0]
print('Generated SQL (first 500 chars):')
print(generated_sql[:500])
print(f'\nTotal SQL length: {len(generated_sql)} chars')
print(f'Result rows:     {top_assets.count()}')

## 3. The data quality problem, stated out loud

Consumption data in building portfolios is typically noisy and sparse. The model says so
rather than smoothing it over. Only about 60% of assets have a smart meter at all, and among
those that do, coverage varies: some meters have been reporting intermittently for months.

Where meter coverage is poor, consumption falls back to the utility invoice. Where the
invoice itself is a supplier estimate, that is flagged too. A platform that quietly presents
an estimate as a measurement cannot be the basis of a client's regulatory reporting.

In [ ]:
SELECT CASE WHEN NOT is_metered              THEN 'No meter, invoices only'
            WHEN data_completeness_pct >= 98  THEN 'Metered, complete'
            WHEN data_completeness_pct >= 80  THEN 'Metered, partial'
            ELSE                                  'Metered, sparse'
       END                                        AS coverage_band,
       COUNT(*)                                    AS asset_months,
       COUNT(DISTINCT asset_id)                    AS assets,
       ROUND(AVG(data_completeness_pct), 1)        AS avg_completeness_pct,
       SUM(CASE WHEN is_estimated THEN 1 ELSE 0 END) AS estimated_months
FROM   FACT_ENERGY_MONTHLY
GROUP  BY coverage_band
ORDER  BY asset_months DESC

In [ ]:
# Coverage is not uniform, which is the point: an average would hide it.
df = cells.data_quality.to_pandas()
fig = px.bar(
    df.sort_values('ASSET_MONTHS', ascending=True),
    x='ASSET_MONTHS', y='COVERAGE_BAND', orientation='h',
    title='Meter coverage across asset-months (2023-01 to 2026-06)',
    labels={'ASSET_MONTHS': 'Asset-months', 'COVERAGE_BAND': ''},
    text='ASSET_MONTHS',
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=340, margin=dict(l=10, r=60, t=50, b=10))
fig.show()

The same aggregation in Snowpark, for comparison. If your team writes PySpark today, this
will read as familiar: `with_column`, `group_by`, `agg`, `sort`. What is different is what
happens underneath — this becomes one SQL statement on a warehouse, with no cluster to size,
start or keep warm.

In [ ]:
fact = session.table('FACT_ENERGY_MONTHLY')

coverage_band = (
    F.when(~F.col('IS_METERED'),                    F.lit('No meter, invoices only'))
     .when(F.col('DATA_COMPLETENESS_PCT') >= 98,    F.lit('Metered, complete'))
     .when(F.col('DATA_COMPLETENESS_PCT') >= 80,    F.lit('Metered, partial'))
     .otherwise(                                    F.lit('Metered, sparse'))
)

(fact.with_column('COVERAGE_BAND', coverage_band)
     .group_by('COVERAGE_BAND')
     .agg(
         F.count('*').alias('ASSET_MONTHS'),
         F.count_distinct('ASSET_ID').alias('ASSETS'),
         F.round(F.avg('DATA_COMPLETENESS_PCT'), 1).alias('AVG_COMPLETENESS_PCT'),
     )
     .sort(F.col('ASSET_MONTHS').desc())
     .show())

## 4. Seasonality, and why the carrier matters more than the kWh

Heating carriers swing hard across the year while electricity is comparatively flat. That
matters for carbon because a kWh of gas and a kWh of electricity are not equivalent, and the
electricity factor varies enormously by country.

In [ ]:
SELECT DATE_TRUNC('month', i.billing_month) AS billing_month,
       i.energy_carrier,
       ROUND(SUM(i.consumption_kwh) / 1e6, 2) AS gwh
FROM   RAW_UTILITY_INVOICES i
WHERE  i.billing_month < DATE '2026-01-01'
GROUP  BY 1, 2
ORDER  BY 1, 2

In [ ]:
df = cells.seasonality.to_pandas()
fig = px.line(
    df, x='BILLING_MONTH', y='GWH', color='ENERGY_CARRIER',
    title='Monthly consumption by energy carrier',
    labels={'GWH': 'Consumption (GWh)', 'BILLING_MONTH': '', 'ENERGY_CARRIER': 'Carrier'},
)
fig.update_layout(height=380, margin=dict(l=10, r=10, t=50, b=10))
fig.show()

The grid factor argument is central to any cross-border portfolio:
identical consumption in France and Germany produces very different carbon.

In [ ]:
SELECT a.asset_country,
       MAX(p.grid_carbon_factor)                                          AS grid_kgco2e_per_kwh,
       ROUND(SUM(f.energy_kwh) / SUM(a.floor_area_m2), 1)                 AS kwh_per_m2,
       ROUND(SUM(f.co2e_kg)   / SUM(a.floor_area_m2), 1)                  AS kgco2e_per_m2
FROM   FACT_ENERGY_MONTHLY f
JOIN   DIM_ASSET a ON a.asset_id = f.asset_id
JOIN   DIM_CRREM_PATHWAY p
       ON  p.pathway_country = a.asset_country
       AND p.pathway_sector  = a.asset_sector
       AND p.pathway_year    = f.reading_year
WHERE  a.asset_sector = 'Office' AND f.reading_year = 2025
GROUP  BY a.asset_country
ORDER  BY kgco2e_per_m2 DESC

## 5. Why the pathway comparison is materialised

Deciding whether an asset is off its pathway is an aggregate-then-compare: sum a year of
carbon, divide by floor area, then compare against a threshold that varies by country,
sector and year. Expressing that as a semantic metric is fragile, and asking an agent to
reconstruct it on every question invites subtly different answers to the same question.

`ASSET_YEAR_PERFORMANCE` resolves it once. Every downstream metric then becomes a plain SUM,
which is what text-to-SQL is reliable at.

In [ ]:
SELECT p.reading_year,
       COUNT(*)                                                    AS asset_years,
       SUM(CASE WHEN p.is_off_pathway THEN 1 ELSE 0 END)           AS off_pathway,
       ROUND(SUM(p.stranded_floor_area_m2))                        AS stranded_m2,
       ROUND(100.0 * SUM(p.stranded_floor_area_m2) / SUM(p.floor_area_m2), 1) AS stranded_pct
FROM   ASSET_YEAR_PERFORMANCE p
GROUP  BY p.reading_year
ORDER  BY p.reading_year

Here is that aggregate-then-compare written out, so the mechanics are visible rather than
implied. Three steps: roll monthly consumption up to asset-year, join the threshold that
applies to that country, sector and year, then compare.

Note the join condition. The threshold is not a single number — it is a three-part key, and
getting it wrong is the most likely way to produce a confidently wrong stranded-asset figure.
This is exactly the kind of rule that should live in one governed place rather than being
re-implemented in every script that needs it.

In [ ]:
asset   = session.table('DIM_ASSET')
pathway = session.table('DIM_CRREM_PATHWAY')

# 1. Monthly consumption up to asset-year.
annual = (
    fact.join(asset, fact['ASSET_ID'] == asset['ASSET_ID'])
        .group_by(asset['ASSET_ID'], fact['READING_YEAR'], asset['FLOOR_AREA_M2'],
                  asset['ASSET_COUNTRY'], asset['ASSET_SECTOR'])
        .agg(F.sum(fact['ENERGY_KWH']).alias('ENERGY_KWH'),
             F.sum(fact['CO2E_KG']).alias('CO2E_KG'))
)

# 2. Attach the threshold on the full three-part key: country, sector, year.
# 3. Compare intensity against it.
scored = (
    annual.join(
        pathway,
        (annual['ASSET_COUNTRY'] == pathway['PATHWAY_COUNTRY'])
        & (annual['ASSET_SECTOR'] == pathway['PATHWAY_SECTOR'])
        & (annual['READING_YEAR'] == pathway['PATHWAY_YEAR']),
    )
    .with_column('CO2E_KG_PER_M2',
                 F.round(F.col('CO2E_KG') / F.col('FLOOR_AREA_M2'), 2))
    .with_column('IS_OFF_PATHWAY',
                 F.col('CO2E_KG_PER_M2') > F.col('THRESHOLD_KGCO2E_PER_M2'))
)

(scored.group_by('READING_YEAR')
       .agg(
           F.round(F.sum(F.when(F.col('IS_OFF_PATHWAY'), F.col('FLOOR_AREA_M2'))
                          .otherwise(F.lit(0)))).alias('STRANDED_M2'),
           F.round(F.sum('FLOOR_AREA_M2')).alias('MONITORED_M2'),
       )
       .sort('READING_YEAR')
       .show())

The 2025 figure matches the materialised table exactly, at 5,748,824 m². 2026 looks small
only because the dataset stops in June, so it carries half a year of consumption against a
full-year threshold — a reminder that partial periods need handling explicitly rather than
being charted next to complete ones.

The share rises year on year for two compounding reasons: the CRREM threshold tightens by
about 6% annually, and the assets themselves drift slightly worse without intervention.
This is the mechanism behind the phrase *stranded asset*.

## 6. Two intensity definitions, one model

This is a common problem in real-estate ESG: an office client wants consumption per
square metre, a nursing home operator wants it per occupied room, and holding both in a rigid
model is what makes it hard. Both live in the semantic view, and the agent is instructed
which is appropriate per sector.

In [ ]:
SELECT a.asset_sector,
       COUNT(*)                                                         AS assets,
       ROUND(SUM(p.energy_kwh) / SUM(p.floor_area_m2), 1)                AS kwh_per_m2,
       CASE WHEN SUM(p.occupied_units) > 0
            THEN ROUND(SUM(p.energy_kwh) / SUM(p.occupied_units))
       END                                                              AS kwh_per_occupied_unit,
       ROUND(SUM(p.co2e_kg) / SUM(p.floor_area_m2), 1)                   AS kgco2e_per_m2
FROM   ASSET_YEAR_PERFORMANCE p
JOIN   DIM_ASSET a ON a.asset_id = p.asset_id
WHERE  p.reading_year = 2025
GROUP  BY a.asset_sector
ORDER  BY kwh_per_m2 DESC

Per square metre, a nursing home looks bad. Per occupied room, the comparison against an
office becomes meaningful, because the thing being serviced is the room and its resident,
not the floor plate. Judging a care home on m² alone is how you end up recommending the
wrong retrofit.

## 7. Why this is different from running the same Python on a lakehouse

A fair question, and one worth answering precisely rather than with a feature list. Most ESG
platforms have two reference points: a hand-built data lake, and a lakehouse (Spark/Databricks).
The comparison is different for each.

### Against the hand-built lake (parquet on S3, a script per use case)

This is the comparison that applies to many ESG platforms today, and it is not a
close one.

| Today | Here |
|---|---|
| Daily dump from Mongo backups to parquet, versioned by convention | Tables with types, keys, comments and time travel |
| A pipeline per use case, each with its own copy of the rules | One transformation, referenced everywhere |
| The CRREM join re-implemented wherever it is needed | Defined once; every consumer inherits it |
| Isolation enforced in application code, per access path | One policy on the table, inherited by every path |
| File layout, small files and compaction are your problem | Not a thing you operate |
| Only the R&D team can answer a question | An agent any energy manager can ask |

### Against a lakehouse

Databricks is a capable platform and this is not a scorecard. Three differences are real and
worth stating plainly:

**1. There is no cluster in this picture.** Everything above compiled to SQL and ran on a
warehouse that suspends when idle. Nothing to size, no executor memory to tune, no cold start
to wait out, no notebook-attached compute to remember to kill. For teams where
machine availability is a recurring bottleneck, removing capacity planning from the
analytics path is worth something concrete.

**2. Governance is attached to the data, not to the access path.** This is the one that matters
for the ISO and SOC commitments, and it is demonstrated below rather than asserted: the same
Snowpark DataFrame returns different rows depending on who is asking, with no filtering code in
the Python at all. Unity Catalog can express row filters too, so the difference is not that
one platform has row-level security. The difference is that here the policy sits on the table,
so SQL, Python, the semantic view, the agent and any BI tool all inherit the same rule, and
there is no access path that can be added later which quietly bypasses it.

**3. Transformation and serving are the same engine.** The semantic view and the agent read the
tables these transformations wrote. There is no publish step into a separate serving layer, and
therefore no window where the two disagree.

### Where this argument stops

Being straight about the limits is more useful than overselling:

- **Snowpark is not a drop-in PySpark replacement.** The idioms are close enough to transfer
  the skill, but the API surface is not identical and a large existing PySpark codebase will
  need work. If running actual PySpark unchanged matters, Snowpark Connect for Spark is the
  honest answer to look at, not this notebook.
- **Heavy custom JVM or Scala work, or distributed deep-learning training loops**, are a
  different conversation. Nothing above is that.
- **This does not remove the need for data engineers.** It removes the need for them to answer
  the same question by hand every week.

### Governance, demonstrated

The same DataFrame expression, run under three different roles. The Python is byte-for-byte
identical each time — the only thing that changes is who is asking. There is no `WHERE
client_id = ...` anywhere in this code, and there is no way for the caller to opt out of it.

In [ ]:
# One expression, defined once, with no tenant filter of any kind in it.
def stranded_by_client(sess):
    return (sess.table('CLIENT_YEAR_EXPOSURE')
                .filter(F.col('REPORTING_YEAR') == 2025)
                .group_by('CLIENT_ID')
                .agg(F.round(F.sum('STRANDED_M2')).alias('STRANDED_M2')))

for role in ['DEEPKI_INTERNAL_ANALYST', 'DEEPKI_TENANT_RHENUS', 'DEEPKI_TENANT_VANEAU']:
    session.sql(f'USE ROLE {role}').collect()
    df = stranded_by_client(session)
    clients = df.count()
    total = df.agg(F.sum('STRANDED_M2')).collect()[0][0]
    print(f'{role:<26} clients visible: {clients:>2}   stranded m2: {int(total or 0):>10,}')

session.sql('USE ROLE ACCOUNTADMIN').collect()
print('\nSame Python. Same expression. The engine decided what each caller could see.')

Worth pausing on what that output means for a multi-tenant SaaS. The isolation rule was written
once, in SQL, by whoever owns governance. It was not written by the person who wrote this
Python, and it cannot be forgotten by the next person who adds a new query. That is a different
risk profile from enforcing tenancy in application code, which is where most platforms
serving 600 clients end up carrying the exposure.

### Putting the Python into production

The transformation above does not have to stay in a notebook. The same Snowpark code can be
deployed as a stored procedure and scheduled with a task, which is the direct replacement for a
cron'd script on a box:

```python
session.sproc.register(
    func=build_dim_asset,              # a plain Python function taking (session)
    name='BUILD_DIM_ASSET',
    packages=['snowflake-snowpark-python'],
    is_permanent=True,
    stage_location='@CUSTOM_DEMOS.DEEPKI.PROC_STAGE',
    replace=True,
)
```

```sql
CREATE OR REPLACE TASK REFRESH_DIM_ASSET
  WAREHOUSE = COMPUTE_WH
  SCHEDULE  = 'USING CRON 0 4 * * * Europe/Paris'
AS CALL CUSTOM_DEMOS.DEEPKI.BUILD_DIM_ASSET();
```

The Python runs inside Snowflake, under the same governance, with the same audit trail as
everything else, and there is no separate scheduler or worker to keep alive. We have not
deployed that here because the demo builds its tables from SQL scripts, but it is the shape
the production version takes.

## 8. Querying the semantic view directly

The agent is not doing anything a person cannot. `SEMANTIC_VIEW()` is available in plain SQL,
so the governed definitions are reusable by dashboards, notebooks and pipelines alike, not
locked inside a chat interface.

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
  CUSTOM_DEMOS.DEEPKI.DEEPKI_PORTFOLIO_SV
  DIMENSIONS client.portfolio_owner, client.energy_manager_address
  METRICS    exposure.client_stranded_area,
             exposure.subscription_value_at_risk,
             exposure.client_stranded_share
  WHERE      exposure.exposure_year = 2025
)
ORDER BY 3 DESC

In [ ]:
df = cells.semantic_query.to_pandas()
fig = px.bar(
    df.sort_values('CLIENT_STRANDED_AREA', ascending=True),
    x='CLIENT_STRANDED_AREA', y='PORTFOLIO_OWNER', orientation='h',
    title='Stranded floor area by client, 2025',
    labels={'CLIENT_STRANDED_AREA': 'Stranded floor area (m2)', 'PORTFOLIO_OWNER': ''},
)
fig.update_layout(showlegend=False, height=460, margin=dict(l=10, r=30, t=50, b=10))
fig.show()

## 9. Calling the agent from SQL

The demo app calls the agent over REST, but the same agent is reachable from SQL with
`DATA_AGENT_RUN`. Worth knowing for two reasons: it makes the agent testable in CI, and it
means scheduled jobs can use it without an HTTP client.

Two gotchas worth knowing: the payload must be a JSON **string**, not an
OBJECT, and the model name must be one the region actually offers.

In [ ]:
import json

question = (
    'Which client drives the largest share of stranded floor area in 2025, '
    'and what is the pattern by sector and heating system?'
)

# DATA_AGENT_RUN takes the payload as a JSON *string*, not an OBJECT.
payload = json.dumps({
    'messages': [
        {'role': 'user', 'content': [{'type': 'text', 'text': question}]}
    ]
})

rows = session.sql(
    'SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('
    "'CUSTOM_DEMOS.DEEPKI.DEEPKI_PORTFOLIO_AGENT', ?) AS response",
    params=[payload],
).collect()

response = json.loads(rows[0]['RESPONSE'])
content = response.get('content', [])

answer = '\n\n'.join(i['text'] for i in content if i.get('type') == 'text')
tools = sorted({
    i['tool_use']['name']
    for i in content
    if i.get('type') == 'tool_use' and i.get('tool_use', {}).get('name')
})

print('Tools the agent chose:', tools)
print()
print(answer)

## 10. Client isolation

For any multi-tenant SaaS platform, isolation between clients is a hard requirement. It is
what underwrites ISO certifications and SOC reports.

One row access policy covers it. The proof is the same query under three roles.

One behaviour to know before relying on this through an agent rather than through SQL:
Cortex Agents resolve permissions from the querying user's **default** role, not the role
active in the session. Switching role mid-session proves the policy on SQL, but a
tenant-scoped agent call needs a user whose default role is the tenant role.

In [ ]:
# The internal analyst sees the whole book; each tenant role sees exactly its own client.
results = []
for role in ['DEEPKI_INTERNAL_ANALYST', 'DEEPKI_TENANT_RHENUS', 'DEEPKI_TENANT_VANEAU']:
    session.sql(f'USE ROLE {role}').collect()
    row = session.sql('''
        SELECT COUNT(*) AS clients_visible,
               ROUND(SUM(stranded_m2)) AS stranded_m2_visible
        FROM   CUSTOM_DEMOS.DEEPKI.CLIENT_YEAR_EXPOSURE
        WHERE  reporting_year = 2025
    ''').collect()[0]
    results.append({
        'acting_role': role,
        'clients_visible': row['CLIENTS_VISIBLE'],
        'stranded_m2_visible': row['STRANDED_M2_VISIBLE'],
    })

session.sql('USE ROLE ACCOUNTADMIN').collect()
pd.DataFrame(results)

The detail worth noticing: `ASSET_YEAR_PERFORMANCE` carries no `client_id` column at all. It
inherits isolation through the join to `DIM_ASSET`, which is where the policy sits. There is
no path to asset-level data that bypasses the filter, and the semantic view enforces the same
join.

## What this replaces

The typical starting point is: database backups, daily parquet dump to object storage, one
script per use case, and the engineering team as the only route to an answer. What is here
instead:

| Then | Now |
|---|---|
| Parquet files on S3, versioned by hand | Tables with types, keys and comments |
| A pipeline per use case | One harmonised model, many questions |
| Metric definitions inside each script | Two intensity definitions in one semantic view |
| Data quality implicit | Completeness and estimation flagged per asset-month |
| Isolation enforced in application code | One row access policy, provable in three queries |
| Answers gated on the R&D team | An agent any energy manager can ask |
| Python and SQL are separate worlds | The same transformation in either, compiling to one engine |

None of this removes the need for data engineers. It moves them off answering the same
question repeatedly by hand.

## Cleanup

This notebook only reads. It creates nothing, so there is nothing to tear down. To remove the
whole demo, drop the schema and the roles it created:

```sql
DROP SCHEMA IF EXISTS CUSTOM_DEMOS.DEEPKI CASCADE;
DROP ROLE IF EXISTS DEEPKI_TENANT_RHENUS;
DROP ROLE IF EXISTS DEEPKI_TENANT_VANEAU;
DROP ROLE IF EXISTS DEEPKI_INTERNAL_ANALYST;
DROP USER IF EXISTS DEEPKI_APP_USER;
```